In [ ]:

import os
import re
import uuid
import shutil
import warnings
import json
from typing import List, Dict, Any
from difflib import SequenceMatcher
import numpy as np

from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter
from langchain_core.documents import Document
from langchain.storage import InMemoryStore
from langchain.retrievers import ParentDocumentRetriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

warnings.filterwarnings("ignore")


class TextNormalizer:
    @staticmethod
    def process(text: str) -> str:
        text = text.translate(str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789"))
        text = text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
        text = re.sub(r"<.*?>", " ", text)
        text = re.sub(r"[#*>]", " ", text)
        text = text.replace("ي", "ی").replace("ك", "ک")
        text = re.sub(r"\s+", " ", text)
        return text.strip()


class MetadataExtractor:
    @staticmethod
    def _normalize(text: str) -> str:
        t = text.replace("ي", "ی").replace("ك", "ک")
        t = t.translate(str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789"))
        t = t.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
        t = re.sub(r"\s+", " ", t).strip()
        return t

    @staticmethod
    def extract(filename: str, text: str) -> Dict[str, Any]:
        clean = MetadataExtractor._normalize(text)
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

        meta = {
            "source": filename,
            "doc_id": str(uuid.uuid4())[:8],
            "category": "مالیات مستقیم",
            "subject": "",
            "year": "",
            "number": "",
            "related_laws": ""
        }

        if (m := re.search(r"### شماره:\s*(.+)", text)):
            meta["number"] = m.group(1).strip()

        if (m := re.search(r"### موضوع:\s*\*?(.*)", text)):
            meta["subject"] = m.group(1).strip()
        else:
            for line in lines[:3]:
                if 5 <= len(line) <= 150:
                    meta["subject"] = line
                    break

        
        if (m := re.search(r"### تاریخ:\s*(\d{4})/\d{1,2}/\d{1,2}", text)):
            meta["year"] = m.group(1)

        t = clean.lower()
        if "ارزش افزوده" in t or "vat" in t or "عوارض" in t:
            meta["category"] = "مالیات بر ارزش افزوده"
        elif any(k in t for k in ["حقوق", "دستمزد", "کارکنان", "مالیات حقوق"]):
            meta["category"] = "مالیات حقوق"
        elif any(k in t for k in ["ماده 169", "جرایم", "ابلاغ", "اعتراض"]):
            meta["category"] = "مقررات اجرایی"

        laws = []
        for law_match in re.findall(r"\[([^\]]+)\]\((https?://inta\.tax\.gov\.ir/Pages/Action/LawsShow/[^\)]+)\)", text):
            law_name, law_link = law_match
            laws.append({law_name: law_link})
        if laws:
            meta["related_laws"] = json.dumps(laws, ensure_ascii=False)

        return meta

class SplitterFactory:
    @staticmethod
    def get_child_splitter(embeddings):
        return RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)



class IntentRouter:
    categories = {
        "مالیات بر ارزش افزوده": ["ارزش افزوده", "VAT", "عوارض"],
        "مالیات حقوق": ["حقوق کارکنان", "مطالبه حقوق", "کسر مالیات", "اعتراض به مالیات حقوق"],
        "مقررات اجرایی": ["جرایم", "ابلاغ", "اعتراض", "کمیسیون", "هیأت حل اختلاف"],
        "مالیات مستقیم": ["اظهارنامه", "عملکرد", "سود", "ترازنامه"],
    }

    def __init__(self, embeddings):
        self.embeddings = embeddings
        self.vectors = {c: np.mean(self.embeddings.embed_documents(kws), axis=0)
                        for c, kws in self.categories.items()}

    def route(self, query: str):
        qv = self.embeddings.embed_query(query)
        best_cat, best_score = None, -1
        q_norm = qv / np.linalg.norm(qv) if np.linalg.norm(qv) else qv
        
        for cat, vec in self.vectors.items():
            vec_norm = vec / np.linalg.norm(vec) if np.linalg.norm(vec) else vec
            score = np.dot(q_norm, vec_norm)
            if score > best_score:
                best_cat, best_score = cat, score
        return best_cat, float(best_score)



class TaxAssistantEngine:
    def __init__(self, folder):
        try:
            print("Downloading Multilingual Embeddings...")
            self.embeddings = HuggingFaceEmbeddings(
                model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
                model_kwargs={'device': 'cpu'}, 
                encode_kwargs={'normalize_embeddings': True} 
            )
        except Exception as e:
             print(f"❌ خطای Embeddings: {e}")
             raise e
             
        self.router = IntentRouter(self.embeddings)
        self.docs_cache = []
        self.retriever = None
        self.vectorstore = None
        self.store = None
        self.data_folder = folder
        self._ingest()

    def _ingest(self):
        if not os.path.exists(self.data_folder):
            print(f"❌ Data folder not found: {self.data_folder}")
            return
        db_path = "./chroma_db"
        
        if os.path.exists(db_path):
            try:
                shutil.rmtree(db_path)
            except PermissionError:
                print("⚠️ Close other processes using ./chroma_db and try running as administrator.")
                return

        for name in os.listdir(self.data_folder):
            if name.endswith(".md"):
                path = os.path.join(self.data_folder, name)
                try:
                    text = open(path, "r", encoding="utf-8").read()
                    clean = TextNormalizer.process(text)
                    meta = MetadataExtractor.extract(name, text)
                    self.docs_cache.append(Document(page_content=clean, metadata=meta))
                except Exception as e:
                    print(f"⚠️ Failed to process file {name}: {e}")
                    
        if not self.docs_cache:
            print("⚠️ No documents found.")
            return

        child = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
        parent = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100)

        self.vectorstore = Chroma(collection_name="tax_docs", embedding_function=self.embeddings, persist_directory=db_path)
        self.store = InMemoryStore()

        self.retriever = ParentDocumentRetriever(
            vectorstore=self.vectorstore,
            docstore=self.store,
            child_splitter=child,
            parent_splitter=parent,
        )
        self.retriever.add_documents(self.docs_cache)
        print(f"✅ Indexed {len(self.docs_cache)} documents.")

    def search(self, query: str, filt: Dict[str, str] = None):
        filter_expr = None
        
        cat, score = self.router.route(query)
        print(f"DEBUG: Intent Routing found {cat} with score {score:.2f}")

        if not filt and score > 0.9:
            filter_expr = {"category": cat}
            print(f"DEBUG: Setting Filter by Intent: {filter_expr}")


        if filt and self.retriever:
            print(f"DEBUG: User Filter (filt) is present: {filt}")
            pre_filtered_docs = self.docs_cache
            
            for key, val in filt.items():
                if key != "subject":
                    pre_filtered_docs = [
                        d for d in pre_filtered_docs 
                        if str(d.metadata.get(key, "")).strip() == val.strip()
                    ]
                    print(f"DEBUG: Filter '{key}' applied. Docs remaining: {len(pre_filtered_docs)}")

            if subject_val := filt.get("subject"):
                pre_filtered_docs = [
                    d for d in pre_filtered_docs 
                    if self._fuzzy_match(d.metadata.get("subject", ""), subject_val, threshold=0.7)
                ]
                print(f"DEBUG: Filter 'subject' applied. Docs remaining: {len(pre_filtered_docs)}")


            if not pre_filtered_docs:
                print("DEBUG: Pre-filtering resulted in zero documents.")
                return [], f"No Meta Match for filter {filt}"
            
            ids = [d.metadata["doc_id"] for d in pre_filtered_docs]
            filter_expr = {"_source_id": {"$in": ids}} 
            print(f"DEBUG: Final Chroma Filter Set via User Filter. IDs count: {len(ids)}")


        print(f"DEBUG: Final filter_expr status (None means Full Search): {filter_expr is not None}")
        
        if filter_expr and self.retriever:
            child_hits = self.vectorstore.similarity_search(query, k=5, filter=filter_expr)
            
            parent_ids = list(set(d.metadata.get("_source_id") for d in child_hits))
            print(f"DEBUG: Chroma hits: {len(child_hits)}, Unique Parent IDs: {len(parent_ids)}")
            
            return [d for d in self.store.mget(parent_ids) if d], "OK (Filtered)"
        
        return self.retriever.invoke(query), "OK (Full)"

    @staticmethod
    def _fuzzy_match(a: str, b: str, threshold: float = 0.5):
        """مقایسه فازی دو رشته"""
        return SequenceMatcher(None, a.strip().lower(), b.strip().lower()).ratio() >= threshold



def display_menu():
    print("\n" + "="*40)
    print("🤖 دستیار هوشمند مالیاتی")
    print("="*40)
    print("1. 🔎 جستجوی هوشمند")
    print("2. 📂 مرور اسناد")
    print("3. ❌ خروج")
    print("="*40)


def main():
    data_path = "./data"
    engine = TaxAssistantEngine(folder=data_path)
    if not engine.docs_cache or not engine.retriever:
        print("⚠️ سیستم به دلیل عدم لود اسناد یا خطای ایندکس‌سازی، متوقف شد.")
        return
    
    valid_fields = ["year", "category", "subject", "number"]
    
    while True:
        display_menu()
        choice = input("انتخاب: ").strip()
        
        if choice == "1":
            query = input("❓ سوال: ").strip()
            filt = {}

            if input("آیا مایل به افزودن فیلتر هستید؟ (y/n): ").lower() == "y":
                print("انتخاب ویژگی‌های فیلتر:")
                for i, field in enumerate(valid_fields, 1):
                    print(f"{i}. {field}")
                
                sel_fields_raw = input("شماره ویژگی‌ها را با کاما وارد کنید (مثال: 1,3): ").split(",")
                
                try:
                    sel_fields = [int(idx.strip()) for idx in sel_fields_raw if idx.strip().isdigit()]
                    
                    for idx in sel_fields:
                        if 1 <= idx <= len(valid_fields):
                            field = valid_fields[idx - 1]
                            val = input(f"مقدار برای {field} ({valid_fields[idx-1]}): ").strip()
                            if val:
                                filt[field] = val
                        else:
                             print(f"⚠️ شماره {idx} نامعتبر است و نادیده گرفته شد.")

                except ValueError:
                    print("⚠️ ورودی شماره‌های فیلتر نامعتبر است.")
                    continue

            print(f"DEBUG_MAIN: Final filter dict sent to search: {filt}")

            results, status = engine.search(query, filt)
            
            if results:
                d = results[0]
                print(f"\n✅ منبع: {d.metadata.get('source')}")
                print(f"سال: {d.metadata.get('year')}")
                print(f"دسته: {d.metadata.get('category')}")
                print(f"موضوع: {d.metadata.get('subject')}")
                print(f"شماره: {d.metadata.get('number')}")
                print(f"وضعیت فیلتر: {status}") 
                print("-"*40)
                print(d.page_content[:600], "...")
            else:
                print(f"⚠️ نتیجه‌ای یافت نشد. وضعیت: {status}")
            input("\n⏎ برای بازگشت Enter...")

        elif choice == "2":
            for d in engine.docs_cache:
                print(f"🔹 منبع: {d.metadata.get('source')}")
                print(f"سال: {d.metadata.get('year')}")
                print(f"دسته: {d.metadata.get('category')}")
                print(f"موضوع: {d.metadata.get('subject')}")
                print(f"شماره: {d.metadata.get('number')}")
                print("-"*50)
            input("\n⏎ برای بازگشت Enter...")

        elif choice == "3":
            print("👋 خداحافظ!")
            break
        else:
            print("⚠️ انتخاب نامعتبر. دوباره تلاش کنید.")


if __name__ == "__main__":
    main()

✅ Indexed 18 documents.

🤖 دستیار هوشمند مالیاتی
1. 🔎 جستجوی هوشمند
2. 📂 مرور اسناد
3. ❌ خروج
⚠️ انتخاب نامعتبر. دوباره تلاش کنید.

🤖 دستیار هوشمند مالیاتی
1. 🔎 جستجوی هوشمند
2. 📂 مرور اسناد
3. ❌ خروج
DEBUG_MAIN: Final filter dict sent to search: {}
DEBUG: Intent Routing found مالیات مستقیم with score 0.70
DEBUG: Final filter_expr status (None means Full Search): False

✅ منبع: بخشنامه_30_4_4562_22052.md
سال: 1377
دسته: مالیات حقوق
موضوع: در مورد معافيت موسسات انتشاراتي -مجله و روزنامه
شماره: 30/4/4562/22052
وضعیت فیلتر: OK (Full)
----------------------------------------
[بخشنامه: 30/4/4562/22052](https://inta.tax.gov.ir/Pages/Action/LawsDocShow/1/4/1/132/1221) موضوع: در مورد معافیت موسسات انتشاراتی -مجله و روزنامه تاریخ: 1377/05/07 شماره: 30/4/4562/22052 پیوست: پیرو بخشنامه شماره 514/4/30 مورخ 29/1/1377 و به منظور اجرای صحیح مقررات مربوط به تبصره 6 الحاقی به ماده 132قانون مالیاتهای مستقیم مصوب اسفندماه 1366 (موضوع ماده 35 قانون اصلاح موادی از قانون مالیاتهای مستقیم مصوب 7/2/71 ) و همچ

In [ ]:
import os
import re
import uuid
from typing import Dict, Any

class TextNormalizer:
    @staticmethod
    def process(text: str) -> str:
        text = text.translate(str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789"))
        text = text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
        text = re.sub(r"<.*?>", " ", text)
        text = re.sub(r"[#*>`]", " ", text)
        text = text.replace("ي", "ی").replace("ك", "ک")
        text = re.sub(r"\s+", " ", text)
        return text.strip()


class MetadataExtractor:

    @staticmethod
    def _normalize(text: str) -> str:
        t = text.replace("ي", "ی").replace("ك", "ک")
        t = t.translate(str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789"))
        t = t.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
        t = re.sub(r"\s+", " ", t).strip()
        return t

    @staticmethod
    def extract(filename: str, text: str) -> Dict[str, Any]:
        clean = MetadataExtractor._normalize(text)
        lines = [ln.strip() for ln in clean.splitlines() if ln.strip()]

        meta = {
            "source": filename,
            "doc_id": str(uuid.uuid4())[:8],
            "category": "مالیات مستقیم",
            "title": lines[0] if lines else "",
            "subject": "",  
            "year": None
        }

        if m := re.search(r"(?:سال|تاریخ)[:\-\s]*([12]\d{3})", clean):
            meta["year"] = m.group(1)
        elif m := re.search(r"(\d{4})\s*خورشیدی", clean):
            meta["year"] = m.group(1)

        if m := re.search(r"موضوع[:\-\s]*(.+?)(?:\n|$)", clean):
            meta["subject"] = m.group(1).strip()
        else:
            match = re.search(r"(?:موضوع.*?)(.*?)(?:پیوست|$)", clean, flags=re.DOTALL)
            if match:
                meta["subject"] = match.group(1).strip()
            else:
                meta["subject"] = " ".join(lines[:5])

        t = clean.lower()
        if "ارزش افزوده" in t or "vat" in t or "عوارض" in t:
            meta["category"] = "مالیات بر ارزش افزوده"
        elif any(k in t for k in ["حقوق", "دستمزد", "کارکنان", "مالیات حقوق"]):
            meta["category"] = "مالیات حقوق"
        elif any(k in t for k in ["ماده 169", "جرایم", "ابلاغ", "اعتراض"]):
            meta["category"] = "مقررات اجرایی"

        return meta


def extract_all_metadata(folder="./data"):
    all_meta = []
    for fname in os.listdir(folder):
        if fname.endswith(".md"):
            path = os.path.join(folder, fname)
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()
            meta = MetadataExtractor.extract(fname, text)
            all_meta.append(meta)
    return all_meta

if __name__ == "__main__":
    metas = extract_all_metadata("./data")
    for m in metas:
        print("🔹 منبع:", m["source"])
        print("سال:", m.get("year"))
        print("دسته:", m.get("category"))
        print("موضوع:", m.get("subject"))
        print("-" * 50)


In [ ]:
import re
from pathlib import Path

def extract_all_metadata(folder_path):
    folder = Path(folder_path)
    all_meta = []

    for file_path in folder.glob("*.md"):
        text = file_path.read_text(encoding="utf-8")
        metadata = {
            "source": file_path.name,
            "year": None,
            "category": None,
            "subject": None,
            "number": None,
            "related_laws": []
        }

        match_number = re.search(r"### شماره:\s*(.+)", text)
        if match_number:
            metadata["number"] = match_number.group(1).strip()

        match_subject = re.search(r"### موضوع:\s*\*?(.*)", text)
        if match_subject:
            metadata["subject"] = match_subject.group(1).strip()

        match_date = re.search(r"### تاریخ:\s*(\d{4})/\d{1,2}/\d{1,2}", text)
        if match_date:
            metadata["year"] = match_date.group(1)

        match_category = re.search(r"دسته:\s*(.+)", text)
        if match_category:
            metadata["category"] = match_category.group(1).strip()

        for law_match in re.findall(r"\[([^\]]+)\]\((https?://inta\.tax\.gov\.ir/Pages/Action/LawsShow/[^\)]+)\)", text):
            law_name, law_link = law_match
            metadata["related_laws"].append({law_name: law_link})

        all_meta.append(metadata)

    return all_meta

if __name__ == "__main__":
    metas = extract_all_metadata("./data")  
    for m in metas:
        print("🔹 منبع:", m["source"])
        print("سال:", m.get("year"))
        print("دسته:", m.get("category"))
        print("موضوع:", m.get("subject"))
        print("شماره:", m.get("number"))
        print("مواد قانونی:", m.get("related_laws"))
        print("-" * 50)


In [ ]:
def main():
    engine = TaxAssistantEngine()

    if not engine.docs_cache:
        print("❌ No data found.")
        return

    while True:
        clear_output(wait=True)
        print("=" * 40)
        print("🤖 دستیار هوشمند مالیاتی (RAG Pro)")
        print("=" * 40)
        print("1. 🔎 جستجوی هوشمند")
        print("2. 📂 مرور اسناد")
        print("3. ❌ خروج")
        print("=" * 40)

        choice = input("انتخاب: ")

        if choice == "1":
            query = input("❓ سوال: ")

            print("🔍 تحلیل دسته...")
            cat, score = engine.router.route(query)
            print(f"📌 دسته: {cat} | امتیاز: {score:.2f}")

            flt = None
            if input("فیلتر؟ (y/n): ").lower() == "y":
                print("1. سال\n2. موضوع\n3. شماره بخشنامه")
                f = input("گزینه: ")

                if f == "1":
                    flt = {"year": input("سال: ").strip()}
                elif f == "2":
                    flt = {"subject": input("موضوع: ").strip()}
                elif f == "3":
                    flt = {"circular_id": input("شماره: ").strip()}

            print("🚀 جستجو...")
            results, status = engine.search(query, flt)

            if results:
                d = results[0]
                print("\n📄 بهترین سند:")
                print("-" * 40)
                print("منبع:", d.metadata.get("source"))
                print("سال:", d.metadata.get("year"))
                print("دسته:", d.metadata.get("category"))
                print("-" * 40)
                print(d.page_content[:500], "...")
            else:
                print("⚠️ نتیجه‌ای یافت نشد.")

            input("\n⏎ ادامه...")

       
        elif choice == "2":
            print("\n📂 فیلتر:")
            print("1. سال")
            print("2. موضوع")
            print("3. دسته")

            f = input("انتخاب: ")

            if f == "1":
                flt = {"year": input("سال: ").strip()}
            elif f == "2":
                flt = {"subject": input("موضوع: ").strip()}
            else:
                flt = {"category": input("دسته: ").strip()}

            ids = engine.find_by_meta(flt)

            if not ids:
                print("❌ پیدا نشد.")
            else:
                print(f"✅ {len(ids)} سند:")
                docs = [d for d in engine.store.mget(ids) if d]
                for i, d in enumerate(docs[:3]):
                    print(f"\n--- سند {i+1} ---")
                    print("موضوع:", d.metadata.get("subject"))
                    print("سال:", d.metadata.get("year"))
                    print("دسته:", d.metadata.get("category"))

            input("\n⏎ ادامه...")

        else:
            break

main()


In [ ]:
#  (بخش 2-4 تسک)

print("🔎 تست فیلتر سال 1380:")
ask_rag("مالیات حقوق", year_filter="1380")

print("\n" + "="*30 + "\n")

print("🔎 تست بدون فیلتر:")
ask_rag("مالیات حقوق")

In [ ]:
available_years = set()
for doc in processed_docs:
    y = doc.metadata.get('year')
    if y:
        available_years.add(y)

print(" سال‌های شناسایی شده در فایل‌ها:")
print(sorted(list(available_years)))

In [ ]:
ask_rag("مالیات حقوق", year_filter="1376") # یا هر سالی که در لیست دیدی
